In [1]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# WORKING DIRECTORY AND SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung1_tissueTumor.root"
tree_name = "t"

selected_volumes = [2, 3, 4, 5]
selected_pdg = 22

# Geant4 process code for Compton scattering
compton_process = 2013

# Incident gamma energy
E0 = 662.0                 # keV

# Electron rest energy
ME_C2 = 510.99895          # keV

# Momentum conversion
# 1 atomic unit of momentum = 3.72738 keV/c
P_AU = 3.72738             # keV/c per a.u.

# Incident gamma direction: source travels along -y
incident_direction = np.array([0.0, -1.0, 0.0])

excel_file = "lung1_tissueTumor_gamma_theta_Q.xlsx"

print("Working directory:", os.getcwd())
print("Input file:", os.path.abspath(file_path))

# =====================================================
# OPEN ROOT FILE
# =====================================================
root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {os.path.abspath(file_path)}")

tree = root_file.Get(tree_name)

if not tree:
    root_file.Close()
    raise KeyError(f"TTree '{tree_name}' not found")

print("Total TTree entries:", tree.GetEntries())

# =====================================================
# SAFE VECTOR READER
# =====================================================
def safe_value(vector, index, default=np.nan):
    """
    Return vector[index] if it exists.
    Otherwise return the specified default.
    """
    if index < len(vector):
        return vector[index]

    return default


# =====================================================
# THETA AND Q CALCULATION
# =====================================================
def calculate_theta_and_q(px, py, pz, e_prime):
    """
    Calculate:
      1. scattering angle theta
      2. theoretical free-electron Compton energy
      3. measured energy transfer
      4. photon momentum transfer q_c
      5. signed longitudinal electron momentum Q

    Energies are in keV.
    q_c and Q are returned in keV/c.
    Q is also returned in atomic units.
    """

    values = [px, py, pz, e_prime]

    if not all(np.isfinite(value) for value in values):
        return {
            "p_magnitude": np.nan,
            "cos_theta": np.nan,
            "theta_rad": np.nan,
            "theta_deg": np.nan,
            "Eprime_measured": np.nan,
            "Eprime_compton_free": np.nan,
            "Et_measured": np.nan,
            "Et_compton_free": np.nan,
            "qc_keV_c": np.nan,
            "Q_keV_c": np.nan,
            "Q_au": np.nan
        }

    # -------------------------------------------------
    # Magnitude of outgoing direction/momentum vector
    # -------------------------------------------------
    p_magnitude = np.sqrt(px**2 + py**2 + pz**2)

    if p_magnitude <= 0:
        return {
            "p_magnitude": p_magnitude,
            "cos_theta": np.nan,
            "theta_rad": np.nan,
            "theta_deg": np.nan,
            "Eprime_measured": e_prime,
            "Eprime_compton_free": np.nan,
            "Et_measured": E0 - e_prime,
            "Et_compton_free": np.nan,
            "qc_keV_c": np.nan,
            "Q_keV_c": np.nan,
            "Q_au": np.nan
        }

    # -------------------------------------------------
    # Scattering angle relative to incident direction
    #
    # incident direction = (0, -1, 0)
    # cos(theta) = -py / |p|
    # -------------------------------------------------
    cos_theta = (
        incident_direction[0] * px
        + incident_direction[1] * py
        + incident_direction[2] * pz
    ) / p_magnitude

    # Avoid numerical values slightly outside [-1, 1]
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    theta_rad = np.arccos(cos_theta)
    theta_deg = np.degrees(theta_rad)

    # -------------------------------------------------
    # Theoretical free-electron Compton scattered energy
    #
    # E' = E0 / [1 + E0/(m_e c^2)(1-cos(theta))]
    # -------------------------------------------------
    e_prime_compton_free = E0 / (
        1.0
        + (E0 / ME_C2) * (1.0 - cos_theta)
    )

    # -------------------------------------------------
    # Energy transfers
    # -------------------------------------------------
    et_measured = E0 - e_prime

    et_compton_free = E0 - e_prime_compton_free

    # -------------------------------------------------
    # Photon momentum transfer
    #
    # q_c = sqrt(E0^2 + E'^2 - 2 E0 E' cos(theta))
    #
    # Since energies are in keV and c is suppressed,
    # q_c is numerically in keV/c.
    # -------------------------------------------------
    qc_squared = (
        E0**2
        + e_prime**2
        - 2.0 * E0 * e_prime * cos_theta
    )

    # Protect against tiny negative values from rounding
    qc_squared = max(qc_squared, 0.0)

    qc_keV_c = np.sqrt(qc_squared)

    # -------------------------------------------------
    # Signed longitudinal momentum
    #
    # Q = (m_e c^2/q_c)
    #     [Et - q_c^2/(2 m_e c^2)]
    # -------------------------------------------------
    if qc_keV_c > 0:
        q_keV_c = (
            ME_C2 / qc_keV_c
        ) * (
            et_measured
            - qc_keV_c**2 / (2.0 * ME_C2)
        )

        q_au = q_keV_c / P_AU

    else:
        q_keV_c = np.nan
        q_au = np.nan

    return {
        "p_magnitude": p_magnitude,
        "cos_theta": cos_theta,
        "theta_rad": theta_rad,
        "theta_deg": theta_deg,
        "Eprime_measured": e_prime,
        "Eprime_compton_free": e_prime_compton_free,
        "Et_measured": et_measured,
        "Et_compton_free": et_compton_free,
        "qc_keV_c": qc_keV_c,
        "Q_keV_c": q_keV_c,
        "Q_au": q_au
    }


# =====================================================
# STORAGE
# =====================================================
volume_data = {
    volume: [] for volume in selected_volumes
}

combined_data = []

column_order = [
    "tree_entry",
    "vector_index",
    "vlm",
    "pdg",
    "pro",
    "is_compton",
    "stp",

    "k",
    "et_root",
    "de",

    "x",
    "y",
    "z",

    "px",
    "py",
    "pz",
    "p_magnitude",

    "cos_theta",
    "theta_rad",
    "theta_deg",

    "E0_keV",
    "Eprime_measured",
    "Eprime_compton_free",

    "Et_measured",
    "Et_compton_free",

    "qc_keV_c",
    "Q_keV_c",
    "Q_au"
]

# =====================================================
# LOOP THROUGH ROOT DATA
# =====================================================
for tree_entry, event in enumerate(tree):

    number_of_records = len(event.pdg)

    for vector_index in range(number_of_records):

        # vlm must exist at this index
        if vector_index >= len(event.vlm):
            continue

        pdg_value = int(event.pdg[vector_index])
        volume_value = int(event.vlm[vector_index])

        # Keep only gamma records
        if pdg_value != selected_pdg:
            continue

        # Keep only selected volumes
        if volume_value not in selected_volumes:
            continue

        pro_value = int(
            safe_value(event.pro, vector_index, -1)
        )

        stp_value = int(
            safe_value(event.stp, vector_index, -1)
        )

        # For a gamma, k is used as E'
        k_value = float(
            safe_value(event.k, vector_index)
        )

        et_root_value = float(
            safe_value(event.et, vector_index)
        )

        de_value = float(
            safe_value(event.de, vector_index)
        )

        x_value = float(
            safe_value(event.x, vector_index)
        )

        y_value = float(
            safe_value(event.y, vector_index)
        )

        z_value = float(
            safe_value(event.z, vector_index)
        )

        px_value = float(
            safe_value(event.px, vector_index)
        )

        py_value = float(
            safe_value(event.py, vector_index)
        )

        pz_value = float(
            safe_value(event.pz, vector_index)
        )

        # Calculate theta and Q
        calculated = calculate_theta_and_q(
            px=px_value,
            py=py_value,
            pz=pz_value,
            e_prime=k_value
        )

        row = {
            "tree_entry": tree_entry,
            "vector_index": vector_index,

            "vlm": volume_value,
            "pdg": pdg_value,
            "pro": pro_value,
            "is_compton": pro_value == compton_process,
            "stp": stp_value,

            "k": k_value,
            "et_root": et_root_value,
            "de": de_value,

            "x": x_value,
            "y": y_value,
            "z": z_value,

            "px": px_value,
            "py": py_value,
            "pz": pz_value,

            "p_magnitude": calculated["p_magnitude"],

            "cos_theta": calculated["cos_theta"],
            "theta_rad": calculated["theta_rad"],
            "theta_deg": calculated["theta_deg"],

            "E0_keV": E0,
            "Eprime_measured": calculated["Eprime_measured"],
            "Eprime_compton_free": calculated[
                "Eprime_compton_free"
            ],

            "Et_measured": calculated["Et_measured"],
            "Et_compton_free": calculated[
                "Et_compton_free"
            ],

            "qc_keV_c": calculated["qc_keV_c"],
            "Q_keV_c": calculated["Q_keV_c"],
            "Q_au": calculated["Q_au"]
        }

        volume_data[volume_value].append(row)
        combined_data.append(row)

# =====================================================
# CREATE DATAFRAMES
# =====================================================
dataframes = {
    volume: pd.DataFrame(
        volume_data[volume],
        columns=column_order
    )
    for volume in selected_volumes
}

df_all = pd.DataFrame(
    combined_data,
    columns=column_order
)

# =====================================================
# CREATE COMPTON-ONLY DATAFRAMES
# =====================================================
compton_dataframes = {
    volume: dataframes[volume][
        dataframes[volume]["is_compton"]
    ].copy()
    for volume in selected_volumes
}

df_compton_all = df_all[
    df_all["is_compton"]
].copy()

# =====================================================
# WRITE EXCEL WORKBOOK
# =====================================================
with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    # All gamma records, separated by volume
    for volume in selected_volumes:

        dataframes[volume].to_excel(
            writer,
            sheet_name=f"Volume_{volume}",
            index=False
        )

    # Combined gamma records
    df_all.to_excel(
        writer,
        sheet_name="All_Volumes",
        index=False
    )

    # Compton-only records for each volume
    for volume in selected_volumes:

        compton_dataframes[volume].to_excel(
            writer,
            sheet_name=f"Compton_Vlm_{volume}",
            index=False
        )

    # All Compton records
    df_compton_all.to_excel(
        writer,
        sheet_name="All_Compton",
        index=False
    )

# =====================================================
# CLOSE ROOT FILE
# =====================================================
root_file.Close()

# =====================================================
# CONFIRM OUTPUT
# =====================================================
print("\nRecords saved:")

for volume in selected_volumes:

    print(
        f"Volume {volume}: "
        f"{len(dataframes[volume])} gamma records, "
        f"{len(compton_dataframes[volume])} Compton records"
    )

print("\nAll selected gamma records:", len(df_all))
print("All selected Compton records:", len(df_compton_all))

print("\nExcel file saved at:")
print(os.path.abspath(excel_file))

print("File exists:", os.path.exists(excel_file))

if os.path.exists(excel_file):
    print(
        "File size:",
        os.path.getsize(excel_file),
        "bytes"
    )

display(df_all.head(10))

Working directory: /root/geant4/detector/Lung_ICRP
Input file: /root/geant4/detector/Lung_ICRP/lung1_tissueTumor.root
Total TTree entries: 50000

Records saved:
Volume 2: 101267 gamma records, 1870 Compton records
Volume 3: 51685 gamma records, 2304 Compton records
Volume 4: 668 gamma records, 160 Compton records
Volume 5: 478 gamma records, 119 Compton records

All selected gamma records: 154098
All selected Compton records: 4453

Excel file saved at:
/root/geant4/detector/Lung_ICRP/lung1_tissueTumor_gamma_theta_Q.xlsx
File exists: True
File size: 30680315 bytes


,tree_entry,vector_index,vlm,pdg,pro,is_compton,stp,k,et_root,de,...,theta_rad,theta_deg,E0_keV,Eprime_measured,Eprime_compton_free,Et_measured,Et_compton_free,qc_keV_c,Q_keV_c,Q_au
0,0,2,2,22,1092,False,2,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
1,0,3,3,22,1092,False,3,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
2,0,4,2,22,1092,False,4,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
3,1,2,2,22,1092,False,2,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
4,1,3,3,22,1092,False,3,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
5,1,4,2,22,1092,False,4,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
6,2,2,2,22,1092,False,2,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
7,2,3,3,22,1092,False,3,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
8,2,4,2,22,1092,False,4,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
9,3,2,2,22,1092,False,2,662.0,NaN,0.0,...,0.0,0.0,662.0,662.0,662.0,0.0,0.0,0.0,NaN,NaN
